In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# Specify the device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# TransE Model Definition
class TransE(nn.Module):
    def __init__(self, entity_count, relation_count, embedding_dim):
        super(TransE, self).__init__()       
        
        self.entity_embedding = nn.Embedding(entity_count, embedding_dim)
        self.relation_embedding = nn.Embedding(relation_count, embedding_dim)
        self.embedding_dim = embedding_dim
        self.initialize_weights()

    def initialize_weights(self):
        nn.init.xavier_uniform_(self.entity_embedding.weight.data)
        nn.init.xavier_uniform_(self.relation_embedding.weight.data)

    def forward(self, head, relation, tail):
        head_emb = self.entity_embedding(head)
        relation_emb = self.relation_embedding(relation)
        tail_emb = self.entity_embedding(tail)
        score = torch.norm(head_emb + relation_emb - tail_emb, p=1, dim=1)
        return score

    def loss(self, pos_score, neg_score, margin=1.0):
        return torch.sum(torch.relu(pos_score - neg_score + margin))

# Dataset Class Definition
class KGDataset(Dataset):
    def __init__(self, triples, entity_count, relation_count):
        self.triples = triples
        self.entity_count = entity_count
        self.relation_count = relation_count

    def __len__(self):
        return len(self.triples)

    def __getitem__(self, idx):
        head, relation, tail = self.triples[idx]
        return torch.tensor(head), torch.tensor(relation), torch.tensor(tail)

    def generate_negative_sample(self, head, relation, tail):
        corrupt_tail = torch.randint(0, 2, (1,)).item() == 1
        if corrupt_tail:
            neg_tail = torch.randint(0, self.entity_count, (1,)).item()
            return head, relation, neg_tail
        else:
            neg_head = torch.randint(0, self.entity_count, (1,)).item()
            return neg_head, relation, tail

# Training Function
def train_transe_model(transE, dataset, batch_size=128, epochs=10, lr=0.001):
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = optim.Adam(transE.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0
        for i, (head, relation, tail) in enumerate(dataloader):
            head, relation, tail = head.to(device), relation.to(device), tail.to(device)

            # Generate negative samples
            neg_head, neg_relation, neg_tail = dataset.generate_negative_sample(head, relation, tail)
            neg_head, neg_relation, neg_tail = (
                torch.tensor(neg_head).to(device),
                torch.tensor(neg_relation).to(device),
                torch.tensor(neg_tail).to(device),
            )

            # Calculate positive and negative scores
            pos_score = transE(head, relation, tail)
            neg_score = transE(neg_head, neg_relation, neg_tail)

            # Calculate loss
            loss = transE.loss(pos_score, neg_score)
            total_loss += loss.item()

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss/len(dataloader)}")

# Function to Load Triples from a File
def load_triples(file_path, entity_count, relation_count):
    triples = []
    with open(file_path, 'r') as file:
        next(file)  # Skip the first line if it contains a header or count
        for line in file:
            head, tail, relation = map(int, line.strip().split())
            assert 0 <= head < entity_count, f"Invalid head entity index: {head}"
            assert 0 <= relation < relation_count, f"Invalid relation index: {relation}"
            assert 0 <= tail < entity_count, f"Invalid tail entity index: {tail}"
            triples.append((head, relation, tail))
    return triples

# Main Execution
entity_count = 14541
relation_count = 237
file_path = "./datasets/benchmarks/FB15K237/train2id.txt"

# Load triples from the file
triples = load_triples(file_path, entity_count=entity_count, relation_count=relation_count)

# Verify the loaded triples
print(f"Loaded {len(triples)} triples.")
print(triples[:5])  # Print the first 5 triples to check

# Initialize the TransE model and move it to the specified device
transE = TransE(entity_count, relation_count, embedding_dim=100).to(device)

# Load dataset
dataset = KGDataset(triples, entity_count, relation_count)

# Train the model on the specified device
train_transe_model(transE, dataset, batch_size=128, epochs=10, lr=0.001)



Loaded 272115 triples.
[(0, 0, 1), (2, 1, 3), (4, 2, 5), (6, 3, 7), (8, 4, 9)]


/tmp/ipykernel_332896/1098162047.py:69: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(neg_head).to(device),
/tmp/ipykernel_332896/1098162047.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(neg_relation).to(device),
/tmp/ipykernel_332896/1098162047.py:71: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(neg_tail).to(device),


Epoch 1/100, Loss: 45.921126476235806
Epoch 2/100, Loss: 13.031304154324285
Epoch 3/100, Loss: 10.470924519528113
Epoch 4/100, Loss: 9.074772081505209
Epoch 5/100, Loss: 8.717271608332924
Epoch 6/100, Loss: 8.415944969553154
Epoch 7/100, Loss: 8.190290265339002
Epoch 8/100, Loss: 7.507082950586658
Epoch 9/100, Loss: 7.623936685434738
Epoch 10/100, Loss: 7.140337449250531
Epoch 11/100, Loss: 6.76652257933666
Epoch 12/100, Loss: 6.855456560373531
Epoch 13/100, Loss: 6.799359265522136
Epoch 14/100, Loss: 6.625735059282405
Epoch 15/100, Loss: 6.252018853166926
Epoch 16/100, Loss: 6.224458578949477
Epoch 17/100, Loss: 5.927049014889061
Epoch 18/100, Loss: 6.261530563950203
Epoch 19/100, Loss: 5.849632995260961
Epoch 20/100, Loss: 5.9556979083386885
Epoch 21/100, Loss: 5.7445729662535445
Epoch 22/100, Loss: 5.867133256521073
Epoch 23/100, Loss: 5.666167692990093
Epoch 24/100, Loss: 5.399109746416223
Epoch 25/100, Loss: 5.629994974199153
Epoch 26/100, Loss: 5.559200126499022
Epoch 27/100, Los

In [2]:


def load_triples(file_path, entity_count, relation_count):
    triples = []
    with open(file_path, 'r') as file:
        next(file)  # Skip the first line if it contains a header or count
        for line in file:
            head, tail, relation = map(int, line.strip().split())
            assert 0 <= head < entity_count, f"Invalid head entity index: {head}"
            assert 0 <= relation < relation_count, f"Invalid relation index: {relation}"
            assert 0 <= tail < entity_count, f"Invalid tail entity index: {tail}"
            triples.append((head, relation, tail))
    return triples


file_path = "./datasets/benchmarks/FB15K237/test2id.txt"

# Load triples from the file
triples = load_triples(file_path, entity_count=entity_count, relation_count=relation_count)

dataset = KGDataset(triples, entity_count, relation_count)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

from data.TestDataLoader import *
import ctypes
from ctypes import c_char_p, POINTER, c_int64, c_float
from tqdm import tqdm

model_name = "FB15K237"

loader = Loader(f"TransE_{model_name}.yaml")
test_dataloader = TestDataLoader(loader) 


lib = ctypes.CDLL("./Base.so")
lib.testHead.argtypes = [ctypes.c_void_p, ctypes.c_int64]
lib.testTail.argtypes = [ctypes.c_void_p, ctypes.c_int64]


for _,  (head, relation, tail) in tqdm(enumerate(dataloader), total=len(dataloader)):
    head = head.to(device)  # Ensure head is loaded correctly
    relation = relation.to(device)  # Ensure relation is loaded correctly
    tail = tail.to(device)  # Ensure tail is loaded correctly
    
    heads = head.repeat(entity_count)
    relations = relation.repeat(entity_count)
    tails = torch.arange(entity_count).to(device) 

    scores_t = transE(heads, relations, tails)
    score_t = scores_t.detach().cpu().numpy()
    lib.testTail(score_t.__array_interface__["data"][0], _)

    heads = torch.arange(entity_count).to(device) 
    relations = relation.repeat(entity_count)
    tails = tail.repeat(entity_count)
    
    scores_h = transE(heads, relations, tails)
    score_h = scores_h.detach().cpu().numpy()
    lib.testHead(score_h.__array_interface__["data"][0], _)

lib.test_link_prediction()



All required files are present in './datasets/benchmarks/FB15K237/'.
Time taken: 0.147817 seconds
C++ Test Data Done!


100%|██████████| 20466/20466 [00:58<00:00, 350.29it/s]

no type constraint results:
metric:			 MRR 		 MR 		 hit@10 	 hit@3  	 hit@1 
H(raw):			 0.001370 	 8110.277832 	 0.002052 	 0.000880 	 0.000244 
T(raw):			 0.005058 	 8970.115234 	 0.007916 	 0.004398 	 0.002834 
averaged(raw):		 0.003214 	 8540.196289 	 0.004984 	 0.002639 	 0.001539 

H(filter):		 0.002302 	 7894.426270 	 0.003860 	 0.001515 	 0.000635 
T(filter):		 0.006277 	 8951.928711 	 0.010945 	 0.005717 	 0.003518 
averaged(filter):	 0.004290 	 8423.177734 	 0.007403 	 0.003616 	 0.002077 


-607462904